# ¿Le falta capacidad a la cabeza de clasificación? Cabeza lineal contra cabeza MLP

**Pregunta:** la cabeza de CPC-GRU es una sola capa lineal (`Linear(256 → 7)`, precedida de LayerNorm y Dropout) sobre `c_T`, el contexto de la GRU en el último instante de la ventana. ¿Una cabeza con una capa oculta no lineal separa mejor las 7 actividades?

**Diseño:** una sola diferencia entre las dos condiciones, todo lo demás idéntico.

| | Cabeza lineal (notebook 11) | Cabeza MLP (este notebook) |
|---|---|---|
| Capas | LayerNorm(256) → Dropout(0.2) → Linear(256, 7) | LayerNorm(256) → Linear(256, 128) → GELU → Dropout(0.3) → Linear(128, 7) |
| Parámetros de la cabeza | 2,311 | 34,311 |

- **Variables:** el subconjunto `EML` (codos, manos y piernas, 10 variables), el mínimo del notebook 11.
- **Mismo preentrenamiento:** en cada ronda se cargan los pesos CPC que guardó el notebook 11 y sólo se repite el LP-FT con la cabeza MLP. Las dos cabezas parten del **mismo encoder y la misma GRU**, así que la diferencia se debe sólo a la cabeza (y no a la variación aleatoria de CPC).
- **Mismo LP-FT:** mismas épocas, paciencias, tasas de aprendizaje, pesos de clase y parada temprana con el sujeto de validación interna.
- **La cabeza lineal no se reentrena:** se toman las 12 rondas de `EML` del notebook 11. Sus pesos guardados se recargan para calcular el F1 en entrenamiento; como control, deben reproducir las predicciones del notebook 11.

**Cómo leer el resultado:**

- **F1 en prueba (LOSO):** si la MLP mejora de forma consistente entre sujetos (Wilcoxon), la capacidad de la cabeza sí limitaba.
- **F1 en entrenamiento:** si la cabeza lineal ya clasifica casi perfecto las ventanas de entrenamiento, no le falta capacidad. La diferencia con la prueba es entonces de **generalización a personas nuevas**, y una cabeza más grande no debería cerrarla.

**Tiempo estimado:** la carga de datos, unos minutos; 12 rondas × (recarga de la cabeza lineal + LP-FT con la MLP) ≈ 30–45 min. Cada ronda se guarda al terminar y se carga si ya existe.

# 1. Configuración

In [ ]:
import copy, json, os, re, time
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from numpy.lib.stride_tricks import sliding_window_view
from scipy.interpolate import interp1d
from scipy.signal import butter, sosfiltfilt
from scipy.stats import wilcoxon
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ----------------------------- Datos y validación (iguales al avance 5) -----------------------------
SEED = 0
TARGET_FS = 12                         # Hz
CV_SUBJECTS = tuple(range(1, 13))      # sujetos de la validación cruzada LOSO
TEST_SUBJECTS = (13, 14)               # reservados para la prueba final: no se usan aquí
WIN, SUP_HOP, CPC_HOP = 48, 6, 24      # ventana de 4 s; una ventana etiquetada cada 0.5 s y una sin etiqueta (CPC) cada 2 s
SCALER_FIT_POINTS = 250_000            # instantes que se muestrean para ajustar el escalador
PROBE_TRAIN_MAX = 15_000               # ventanas de la sonda que decide la parada temprana de CPC (como el notebook 10)

# ----------------------------- CPC-GRU + LP-FT (los valores de los notebooks 10 y 11) -----------------------------
CPC_EPOCHS, CPC_EVAL_EVERY, CPC_PATIENCE = 20, 2, 3
CPC_BATCH, CPC_LR, GRAD_CLIP, N_ANCHORS = 128, 5e-4, 1.0, 4
HORIZONS = tuple(range(1, 13))         # CPC predice de 1 a 12 pasos adelante (1 s)
FINETUNE_BATCH = 256
FINETUNE_LR_HEAD, FINETUNE_LR_BACKBONE = 1e-3, 1e-4
FT_EPOCHS, FT_PATIENCE = 30, 5         # etapa FT de LP-FT
LP_EPOCHS, LP_PATIENCE, LP_LR = 20, 3, 1e-3   # etapa LP de LP-FT
STEPS_CAP = FT_TRAIN_CAP = None        # límites sólo para la corrida de prueba

# ----------------------------- Cabezas que se comparan -----------------------------
HEADS = ("lineal", "mlp")
MLP_HIDDEN, MLP_DROPOUT = 128, 0.3    # cabeza MLP: 256 -> 128 -> 7

# Corrida de prueba rápida (2 rondas, pocas épocas) para revisar que todo funciona: NB13_QUICK=1
QUICK = os.environ.get("NB13_QUICK") == "1"
FOLDS = CV_SUBJECTS
if QUICK:
    FOLDS = (1, 2)
    PROBE_TRAIN_MAX, FT_TRAIN_CAP, LP_EPOCHS, FT_EPOCHS = 2_000, 2_000, 1, 1

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
SYNC_DIR = REPO_DIR / "notebooks" / "_cache" / "sync" / f"fs{TARGET_FS}"                     # exportación MVNX sincronizada
RECOVER_DIR = REPO_DIR / "notebooks" / "_cache" / "sync" / f"fs{TARGET_FS}_mvnx_recuperados"  # caché de los ensayos recuperados
MVNX_DIR, MVNX_LABEL_DIR = REPO_DIR / "datos_unzipped" / "mvnx_files", REPO_DIR / "datos_unzipped" / "Labels 2"
SAVED_DIR = REPO_DIR / "checkpoints" / f"cpc10_loso_mvnx_metodos_fs{TARGET_FS}"   # rondas del notebook 10 (30 variables)
NB11_DIR = REPO_DIR / "checkpoints" / f"nb11_variables_minimas_fs{TARGET_FS}"          # rondas del notebook 11
RUN_DIR = REPO_DIR / "checkpoints" / (f"nb13_cabeza_mlp_fs{TARGET_FS}" + ("_quick" if QUICK else ""))
RUN_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = RUN_DIR / "progreso.log"
assert (SYNC_DIR / "metadata.json").exists(), f"Falta la exportación sincronizada en {SYNC_DIR.relative_to(REPO_DIR)}"
assert SAVED_DIR.exists(), f"Faltan las rondas del notebook 10 en {SAVED_DIR.relative_to(REPO_DIR)}"
assert (NB11_DIR / "gru_ft" / "EML").exists(), f"Faltan las rondas EML del notebook 11 en {NB11_DIR.relative_to(REPO_DIR)}"

SYNC_META = json.loads((SYNC_DIR / "metadata.json").read_text())
LABELS, NO_LABEL = SYNC_META["label_order"], SYNC_META["sin_etiqueta"]
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})


def log(message):
    """Imprime un mensaje con hora y lo guarda en progreso.log (útil para seguir corridas largas)."""
    line = f"{datetime.now():%Y-%m-%d %H:%M} | {message}"
    print(line)
    with open(LOG_FILE, "a") as fh:
        fh.write(line + "\n")


print(f"PyTorch {torch.__version__} | dispositivo: {DEVICE} | rondas: {len(FOLDS)} | QUICK: {QUICK}")
print(f"Resultados en: {RUN_DIR.relative_to(REPO_DIR)}")

# 2. Datos

Los mismos del notebook 11: MVNX a 12 Hz, las 30 variables de Xsens, etiqueta del instante final, ventanas de 4 s y los 3 ensayos recuperados del sujeto 7. No se construye la tabla del Random Forest.

In [ ]:
# Columnas MVNX crudas que se necesitan para calcular las 30 variables
RAW_COLUMNS = (
    [f"position__{s}__{a}" for s in ("Pelvis", "Head", "LeftHand", "RightHand") for a in "xyz"]          # posiciones (m)
    + [f"velocity__{s}__{a}" for s in ("Pelvis", "LeftHand", "RightHand") for a in "xyz"]               # velocidades (m/s)
    + [f"angularVelocity__{s}__{a}" for s in ("Pelvis", "T8") for a in "xyz"]                            # giros (rad/s)
    + [f"jointAngle__{j}__c3" for j in ("jL5S1", "jC1Head", "jLeftElbow", "jRightElbow", "jLeftWrist", "jRightWrist",
                                       "jLeftHip", "jRightHip", "jLeftKnee", "jRightKnee")]               # flexiones (°)
    + [f"jointAngleXZY__{j}__{c}" for j in ("jLeftShoulder", "jRightShoulder") for c in ("c1", "c3")]   # hombros en XZY
    + [f"footContacts__{f}_{p}" for f in ("LeftFoot", "RightFoot") for p in ("Heel", "Toe")]            # contactos (0/1)
)
RAW_IDX = {c: i for i, c in enumerate(RAW_COLUMNS)}   # nombre de columna -> posición
SIDES = (("izq", "Left"), ("der", "Right"))           # prefijo en español y nombre del lado en el MVNX


def derive(raw):
    """Columnas MVNX crudas -> las 30 variables, con los lados fijos (izquierdo / derecho)."""
    col = lambda name: raw[:, RAW_IDX[name]].astype(np.float64)
    # --- Postura (14): ángulos articulares en grados ---
    out = {"tronco_flexion": col("jointAngle__jL5S1__c3"), "cuello_flexion": col("jointAngle__jC1Head__c3")}
    for lado, s in SIDES:
        out[f"{lado}_hombro_flexion"] = col(f"jointAngleXZY__j{s}Shoulder__c3")
        out[f"{lado}_hombro_abduccion"] = col(f"jointAngleXZY__j{s}Shoulder__c1")
        out[f"{lado}_codo_flexion"] = col(f"jointAngle__j{s}Elbow__c3")
        out[f"{lado}_muneca_flexion"] = col(f"jointAngle__j{s}Wrist__c3")
        out[f"{lado}_cadera_flexion"] = col(f"jointAngle__j{s}Hip__c3")
        out[f"{lado}_rodilla_flexion"] = col(f"jointAngle__j{s}Knee__c3")
    # --- Alturas y alcance (6): en metros; el alcance es la distancia horizontal mano–pelvis ---
    out["pelvis_altura"] = col("position__Pelvis__z")
    out["cabeza_altura"] = col("position__Head__z")
    for lado, s in SIDES:
        out[f"{lado}_mano_altura"] = col(f"position__{s}Hand__z")
        out[f"{lado}_mano_alcance"] = np.hypot(col(f"position__{s}Hand__x") - col("position__Pelvis__x"),
                                               col(f"position__{s}Hand__y") - col("position__Pelvis__y"))
    # --- Velocidades (6): normas, para que no dependan de hacia dónde mira la persona ---
    out["pelvis_vel_horizontal"] = np.hypot(col("velocity__Pelvis__x"), col("velocity__Pelvis__y"))
    out["pelvis_vel_vertical"] = col("velocity__Pelvis__z")
    for lado, s in SIDES:
        out[f"{lado}_mano_rapidez"] = np.linalg.norm([col(f"velocity__{s}Hand__{a}") for a in "xyz"], axis=0)
    out["pelvis_giro"] = np.abs(col("angularVelocity__Pelvis__z"))
    out["tronco_vel_angular"] = np.linalg.norm([col(f"angularVelocity__T8__{a}") for a in "xyz"], axis=0)
    # --- Apoyo (4): contacto del talón y la punta de cada pie con el suelo ---
    for lado, s in SIDES:
        out[f"{lado}_talon"] = col(f"footContacts__{s}Foot_Heel")
        out[f"{lado}_punta"] = col(f"footContacts__{s}Foot_Toe")
    return np.column_stack(list(out.values())).astype(np.float32), list(out)


# Unidad y grupo de cada variable (se usan en tablas, en el escalador y en la gráfica de importancia)
def feature_unit(name):
    if name.endswith(("flexion", "abduccion")):
        return "°"
    if name.endswith(("altura", "alcance")):
        return "m"
    if name.endswith(("vel_horizontal", "vel_vertical", "rapidez")):
        return "m/s"
    if name.endswith(("giro", "vel_angular")):
        return "rad/s"
    return "0/1"

In [ ]:
# Ensayos recuperados: los que la exportación excluyó SÓLO por la sEMG se leen del MVNX crudo, con los mismos
# controles de calidad del MVNX y el mismo remuestreo que la exportación.
EXCLUDED = pd.read_csv(SYNC_DIR / "exclusiones.csv")
SEMG_ONLY = EXCLUDED.groupby("mvnx_file")["motivo"].agg(lambda m: m.str.startswith("sEMG").all())   # excluido sólo por sEMG
RECOVER = sorted(SEMG_ONLY[SEMG_ONLY].index)
NS = "{http://www.xsens.com/mvn/mvnx}"                 # espacio de nombres del XML de Xsens
# Campos que se leen y cómo se remuestrea cada uno
FIELDS = {"position": "continua", "velocity": "continua", "angularVelocity": "continua",
          "jointAngle": "angulo", "jointAngleXZY": "angulo", "footContacts": "binaria"}
# Nombre de archivo -> sujeto, tarea, lado, carga y condición
MVNX_RE = re.compile(r"Subj_(?P<subject>\d+)_(?P<task>Isokin|MMH_Bim|MMH_One)_(?:(?P<side>[LR])_)?"
                     r"(?P<load_kg>\d+)kg_(?P<condition>St|\d+bpm|Trial_\d)\.mvnx")


def read_mvnx_fields(path):
    """Tiempos (s) y campos de FIELDS de los frames normales de un MVNX, con los nombres de su cabecera."""
    names = {"segment": [], "joint": [], "contactDefinition": []}
    times, raw = [], {}
    # El XML pesa cientos de MB: se lee por eventos y cada frame se borra después de leerlo
    with open(path, "rb") as fh:
        frames = None
        for event, el in ET.iterparse(fh, events=("start", "end")):
            tag = el.tag[len(NS):] if el.tag.startswith(NS) else el.tag
            if event == "start" and tag == "frames":
                frames = el
            elif event == "end" and tag in names and el.get("label") and frames is None:
                names[tag].append(el.get("label"))                     # nombres de segmentos, articulaciones y contactos
            elif event == "end" and tag == "frame":
                if el.get("type") == "normal":                         # se ignoran los frames de calibración
                    times.append(float(el.get("time")))
                    for c in el:
                        f = c.tag[len(NS):]
                        if f in FIELDS:
                            raw.setdefault(f, []).append(c.text or "")
                frames.remove(el)
    n = len(times)
    data = {k: np.array(" ".join(v).split(), dtype=np.float64).reshape(n, -1) for k, v in raw.items()}
    t = np.asarray(times) / 1000.0
    return t - t[0], data, names


# Nombres de columna de un campo, con el mismo formato que la exportación sincronizada
def mvnx_columns(field, names):
    if field in ("jointAngle", "jointAngleXZY"):
        return [f"{field}__{j}__c{c}" for j in names["joint"] for c in (1, 2, 3)]
    if field == "footContacts":
        return [f"{field}__{c}" for c in names["contactDefinition"]]
    return [f"{field}__{s}__{a}" for s in names["segment"] for a in "xyz"]


# Pasa-bajas antialias de fase cero (Butterworth de orden 4 a 4.8 Hz) antes de bajar de 240 a 12 Hz
def lowpass(x, fs):
    sos = butter(SYNC_META["antialias"]["orden"], SYNC_META["antialias"]["corte_hz"], btype="low", fs=fs, output="sos")
    return sosfiltfilt(sos, x, axis=0)


# Vecino más cercano: para contactos (0/1) y etiquetas, que no se pueden interpolar
def nearest(t, values, grid):
    idx = np.clip(np.searchsorted(t, grid), 1, len(t) - 1)
    idx -= (grid - t[idx - 1]) < (t[idx] - grid)
    return values[idx]


# Remuestrea un campo a la rejilla de 12 Hz según su tipo
def resample_field(t, x, grid, fs, kind):
    if kind == "binaria":
        return nearest(t, x, grid)
    if kind == "angulo":
        x = np.rad2deg(np.unwrap(np.deg2rad(x), axis=0))              # evita saltos de 180° a −180° al filtrar
    y = interp1d(t, lowpass(x, fs), axis=0, assume_sorted=True)(grid)
    return (y + 180.0) % 360.0 - 180.0 if kind == "angulo" else y


# Etiquetas válidas: actividades conocidas, segmentos con inicio antes del fin y sin traslapes
def labels_ok(lab):
    gap = lab["Start_Frame"].shift(-1) - lab["End_Frame"]
    return bool(lab["Label"].isin(LABELS).all() and not ((lab["End_Frame"] < lab["Start_Frame"]).any() or (gap < 0).any()))


def process_mvnx(mvnx_file, grid=None):
    """MVNX crudo -> (columnas RAW_COLUMNS, label_mvnx, tiempos) a TARGET_FS Hz, más sus controles de calidad."""
    subj = f"Subj_{int(MVNX_RE.fullmatch(mvnx_file)['subject']):02d}"
    t, data, names = read_mvnx_fields(MVNX_DIR / subj / mvnx_file)
    lab = pd.read_csv(MVNX_LABEL_DIR / subj / mvnx_file.replace(".mvnx", "_mvnx_labels.csv"))
    fs = (len(t) - 1) / t[-1]
    # Controles de calidad: saltos de tiempo en la grabación y etiquetas consistentes
    checks = {"n_time_gaps": int((np.diff(t) * 1000 > 1000 / 240 + 1.1).sum()), "labels_ok": labels_ok(lab)}
    if grid is None:
        grid = np.arange(int(np.floor(t[-1] * TARGET_FS)) + 1) / TARGET_FS
    cols, blocks = [], []
    for field, kind in FIELDS.items():
        blocks.append(resample_field(t, data[field], grid, fs, kind))
        cols += mvnx_columns(field, names)
    pos = {c: i for i, c in enumerate(cols)}
    mv = np.concatenate(blocks, axis=1)[:, [pos[c] for c in RAW_COLUMNS]].astype(np.float32)
    # Etiqueta de cada frame a partir de los segmentos anotados (en frames de MVNX)
    y = np.full(len(t), NO_LABEL, dtype=np.int8)
    base = SYNC_META["base_indices_mvnx"]
    for s0, s1, name in lab[["Start_Frame", "End_Frame", "Label"]].itertuples(index=False):
        y[max(0, s0 - base):s1 - base + 1] = LABELS.index(name)
    return mv, nearest(t, y, grid), grid, checks


def recovered_trial(mvnx_file):
    """Ensayo recuperado desde su caché, o desde el MVNX crudo la primera vez."""
    RECOVER_DIR.mkdir(parents=True, exist_ok=True)
    out = RECOVER_DIR / mvnx_file.replace(".mvnx", ".npz")
    if out.exists() and out.with_suffix(".json").exists():
        with np.load(out) as z:
            return z["mvnx"], z["label_mvnx"], z["time_s"], json.loads(out.with_suffix(".json").read_text())
    mv, lab, grid, checks = process_mvnx(mvnx_file)
    checks["nan"] = int(np.isnan(mv).sum())
    np.savez_compressed(out, mvnx=mv, label_mvnx=lab, time_s=grid)
    out.with_suffix(".json").write_text(json.dumps(checks, default=float))
    return mv, lab, grid, checks


# Control: la lectura del MVNX crudo reproduce la exportación en un ensayo incluido del mismo sujeto (si hay MVNX crudos)
REFERENCE = "Subj_07_MMH_Bim_08kg_Trial_1.mvnx"
if MVNX_DIR.exists() and not all((RECOVER_DIR / f.replace(".mvnx", ".npz")).exists() for f in RECOVER):
    base = SYNC_DIR / "bimanual"
    r = pd.read_csv(base / "trials.csv").set_index("mvnx_file").loc[REFERENCE]
    pos = {c["name"]: i for i, c in enumerate(json.loads((base / "columns.json").read_text())["mvnx"])}
    sl = slice(r.offset, r.offset + r.n_steps)
    with np.load(base / "series.npz") as z:
        ref_mv, ref_lab, ref_grid = z["mvnx"][sl][:, [pos[c] for c in RAW_COLUMNS]], z["label_mvnx"][sl], z["time_s"][sl]
    mv, lab, _, _ = process_mvnx(REFERENCE, grid=ref_grid)
    assert np.abs(mv - ref_mv).max() < 1e-3 and (lab == ref_lab).all(), "La lectura del MVNX crudo no reproduce la exportación"
    del ref_mv, mv

# Se conservan sólo los recuperados que pasan los mismos controles que el resto del MVNX
recovered = {f: recovered_trial(f) for f in RECOVER}
RECOVERED = [f for f, (_, _, _, c) in recovered.items() if c["n_time_gaps"] == 0 and c["labels_ok"] and c["nan"] == 0]
print(f"Ensayos recuperados que pasan los controles del MVNX: {len(RECOVERED)} de {len(RECOVER)} ({', '.join(RECOVERED)})")

In [ ]:
# Serie completa: todos los ensayos, uno tras otro. `trials` dice qué filas corresponden a cada ensayo.
rows, mv_parts, lab_parts, offsets = [], [], [], [0]
for conjunto in ("un_brazo", "bimanual"):
    # Exportación sincronizada de cada conjunto de ensayos: series a 12 Hz y tabla de ensayos
    base = SYNC_DIR / conjunto
    names = [c["name"] for c in json.loads((base / "columns.json").read_text())["mvnx"]]
    pos = {n: i for i, n in enumerate(names)}
    with np.load(base / "series.npz") as z:
        mvnx, label_mvnx = z["mvnx"][:, [pos[c] for c in RAW_COLUMNS]], z["label_mvnx"]
    for r in pd.read_csv(base / "trials.csv").itertuples(index=False):
        sl = slice(r.offset, r.offset + r.n_steps)
        mv, FEATURES = derive(mvnx[sl])                                # 782 columnas -> 30 variables
        rows.append({"trial_idx": len(rows), "mvnx_file": r.mvnx_file, "conjunto": conjunto, "subject": r.subject,
                     "offset": offsets[-1], "n_steps": r.n_steps, "recuperado": False})
        mv_parts.append(mv); lab_parts.append(label_mvnx[sl])
        offsets.append(offsets[-1] + r.n_steps)
    del mvnx, label_mvnx
# Se agregan al final los ensayos recuperados de la sección anterior
for f in RECOVERED:
    mv_raw, lab, grid, _ = recovered[f]
    mv, FEATURES = derive(mv_raw)
    rows.append({"trial_idx": len(rows), "mvnx_file": f, "conjunto": "bimanual", "subject": int(MVNX_RE.fullmatch(f)["subject"]),
                 "offset": offsets[-1], "n_steps": len(grid), "recuperado": True})
    mv_parts.append(mv); lab_parts.append(lab.astype(np.int64))
    offsets.append(offsets[-1] + len(grid))

trials = pd.DataFrame(rows)
offsets = np.asarray(offsets[:-1])                             # fila donde empieza cada ensayo
signals_raw = np.concatenate(mv_parts).astype(np.float32)      # (instantes, 30), unidades físicas
labels_series = np.concatenate(lab_parts).astype(np.int64)     # (instantes,), índice de LABELS o NO_LABEL
CHANNELS, N_CH = list(FEATURES), len(FEATURES)
BINARY = [i for i, n in enumerate(CHANNELS) if feature_unit(n) == "0/1"]   # contactos: el escalador no los modifica
del mv_parts, lab_parts
assert N_CH == 30 and not np.isnan(signals_raw).any() and trials["mvnx_file"].is_unique

print(f"{len(trials)} ensayos de {trials['subject'].nunique()} sujetos | {len(signals_raw):,} instantes "
      f"({len(signals_raw) / TARGET_FS / 3600:.1f} h) | {N_CH} variables")

In [ ]:
# Índices de ventanas: no se copian los datos, sólo se guarda dónde empieza cada ventana
def build_indices():
    sup, cpc = [], []
    for r in trials.itertuples():
        if r.n_steps < WIN:                                            # ensayo más corto que una ventana
            continue
        lab = labels_series[r.offset:r.offset + r.n_steps]
        # Ventanas sin etiqueta para CPC: una cada CPC_HOP pasos (2 s)
        cpc += [(r.trial_idx, r.subject, s) for s in range(0, r.n_steps - WIN + 1, CPC_HOP)]
        # Ventanas etiquetadas: una cada SUP_HOP pasos (0.5 s), con la etiqueta del instante final; sin etiqueta se descartan
        ends = np.arange(WIN - 1, r.n_steps, SUP_HOP)
        sup += [(r.trial_idx, r.subject, e - WIN + 1, e, lab[e]) for e in ends[lab[ends] != NO_LABEL]]
    return (pd.DataFrame(sup, columns=["trial_idx", "subject", "start_step", "end_step", "target"]),
            pd.DataFrame(cpc, columns=["trial_idx", "subject", "start_step"]))


def starts_of(frame):
    """Fila de `signals_raw` donde empieza cada ventana."""
    return offsets[frame["trial_idx"].to_numpy()] + frame["start_step"].to_numpy()


sup_index, cpc_index = build_indices()
sup_cv = sup_index[sup_index["subject"].isin(CV_SUBJECTS)]     # ventanas etiquetadas de los sujetos 1–12
cpc_cv = cpc_index[cpc_index["subject"].isin(CV_SUBJECTS)]     # ventanas sin etiqueta para CPC
assert not sup_cv["subject"].isin(TEST_SUBJECTS).any() and sup_cv["target"].between(0, len(LABELS) - 1).all()

print(f"Ventanas etiquetadas (sujetos 1–12): {len(sup_cv):,} | ventanas sin etiqueta para CPC: {len(cpc_cv):,}")



# Escalador de las redes: se ajusta en cada ronda sólo con los sujetos de entrenamiento (sin fuga del sujeto evaluado)
def fit_scaler(train_subjects, seed=SEED):
    """Escalador robusto ajustado con los instantes de los sujetos de entrenamiento. Contactos (0/1): sin cambio."""
    rows = trials[trials["subject"].isin(train_subjects)]
    idx = np.concatenate([np.arange(r.offset, r.offset + r.n_steps) for r in rows.itertuples()])
    sample = signals_raw[np.random.default_rng(seed).choice(idx, min(SCALER_FIT_POINTS, len(idx)), replace=False)].astype(np.float64)
    # Recorte en los cuantiles 0.1 % y 99.9 %, luego (x − mediana) / (q95 − q5)
    lower, upper = np.quantile(sample, 0.001, axis=0), np.quantile(sample, 0.999, axis=0)
    x = np.clip(sample, lower, upper)
    q5, center, q95 = np.percentile(x, [5, 50, 95], axis=0)
    scale = np.where(q95 - q5 > 0, q95 - q5, 1.0)
    lower[BINARY], upper[BINARY], center[BINARY], scale[BINARY] = 0.0, 1.0, 0.0, 1.0
    return {k: v.astype(np.float32).tolist() for k, v in {"lower": lower, "upper": upper, "center": center, "scale": scale}.items()}


def apply_scaler(params, x=None):
    x = signals_raw if x is None else x
    lower, upper, center, scale = (np.asarray(params[k], dtype=np.float32) for k in ("lower", "upper", "center", "scale"))
    return ((np.clip(x, lower, upper) - center) / scale).astype(np.float32)

# 3. Subconjunto y protocolo

Los grupos del notebook 11 y el subconjunto `EML` (codos, manos y piernas): 10 variables, 11 sensores. El protocolo es el mismo: LOSO con los sujetos 1–12; en cada ronda el sujeto siguiente es la validación interna y los 10 restantes entrenan.

In [ ]:
SENSORS_ALL = ("Pelvis", "T8", "Head", "LeftShoulder", "RightShoulder", "LeftUpperArm", "RightUpperArm", "LeftForeArm",
               "RightForeArm", "LeftHand", "RightHand", "LeftUpperLeg", "RightUpperLeg", "LeftLowerLeg", "RightLowerLeg",
               "LeftFoot", "RightFoot")
both = lambda *names: [f"{lado}_{n}" for n in names for lado in ("izq", "der")]   # izquierda y derecha siempre juntas
pair = lambda *segments: [f"{side}{s}" for s in segments for side in ("Left", "Right")]

# Código -> nombre, variables y sensores que necesita. El orden es el de la tabla.
GROUPS = {
    "G": {"nombre": "Posiciones globales", "variables": ["pelvis_altura", "cabeza_altura"] + both("mano_altura", "mano_alcance"),
          "sensores": list(SENSORS_ALL)},
    "P": {"nombre": "Pelvis", "variables": ["pelvis_vel_horizontal", "pelvis_vel_vertical", "pelvis_giro"],
          "sensores": ["Pelvis"]},
    "T": {"nombre": "Tronco", "variables": ["tronco_flexion", "tronco_vel_angular"], "sensores": ["Pelvis", "T8"]},
    "C": {"nombre": "Cabeza", "variables": ["cuello_flexion"], "sensores": ["T8", "Head"]},
    "H": {"nombre": "Hombros", "variables": both("hombro_flexion", "hombro_abduccion"),
          "sensores": pair("Shoulder", "UpperArm")},
    "E": {"nombre": "Codos", "variables": both("codo_flexion"), "sensores": pair("UpperArm", "ForeArm")},
    "M": {"nombre": "Manos", "variables": both("muneca_flexion", "mano_rapidez"), "sensores": pair("ForeArm", "Hand")},
    "L": {"nombre": "Piernas", "variables": both("cadera_flexion", "rodilla_flexion"),
          "sensores": ["Pelvis"] + pair("UpperLeg", "LowerLeg")},
    "F": {"nombre": "Pies", "variables": both("talon", "punta"), "sensores": pair("LowerLeg", "Foot")},
}
# Control: los grupos reparten las 30 variables sin repetir ninguna, y sólo usan sensores del traje
grouped = [v for g in GROUPS.values() for v in g["variables"]]
assert sorted(grouped) == sorted(CHANNELS) and len(grouped) == len(set(grouped)), "Los grupos no cubren las 30 variables"
assert all(set(g["sensores"]) <= set(SENSORS_ALL) for g in GROUPS.values())


def subset_key(groups):
    """Nombre corto de un subconjunto: los códigos de sus grupos en el orden de GROUPS (p. ej. 'PTEM')."""
    return "".join(g for g in GROUPS if g in groups)


def variables_of(groups):
    """Variables de un subconjunto, en el orden de CHANNELS."""
    keep = {v for g in groups for v in GROUPS[g]["variables"]}
    return [c for c in CHANNELS if c in keep]


def sensors_of(groups):
    return sorted({s for g in groups for s in GROUPS[g]["sensores"]}, key=SENSORS_ALL.index)



SUBSET = ("E", "M", "L")                 # codos, manos y piernas: el mínimo del notebook 11
VARS = variables_of(SUBSET)
print(f"{subset_key(SUBSET)}: {len(VARS)} variables: {', '.join(VARS)}")

In [ ]:
def inner_validation_subject(held_out):
    """El sujeto siguiente de la lista (de forma circular) es la validación interna de la ronda."""
    i = CV_SUBJECTS.index(held_out)
    return CV_SUBJECTS[(i + 1) % len(CV_SUBJECTS)]


def split_subjects(held_out):
    """(sujeto de validación interna, sujetos de entrenamiento) de la ronda en la que se evalúa `held_out`."""
    inner_val = inner_validation_subject(held_out)
    return inner_val, tuple(s for s in CV_SUBJECTS if s not in (held_out, inner_val))


def eval_set(frame, cap=None):
    """Ventanas de un conjunto. Con `cap`, una muestra estratificada por actividad (la misma del notebook 10)."""
    if cap is not None and len(frame) > cap:
        frame = frame.groupby("target", group_keys=False).sample(frac=cap / len(frame), random_state=SEED)
    return {"index": frame.index.to_numpy(), "starts": starts_of(frame), "y": frame["target"].to_numpy()}


def make_sets(held_out):
    """Conjuntos de la ronda: sonda (parada temprana de CPC), ajuste fino, validación interna y sujeto evaluado."""
    inner_val, train_subjects = split_subjects(held_out)
    train = sup_cv[sup_cv["subject"].isin(train_subjects)]
    sets = {"probe_train": eval_set(train, PROBE_TRAIN_MAX), "ft_train": eval_set(train, FT_TRAIN_CAP),
            "val": eval_set(sup_cv[sup_cv["subject"] == inner_val]),
            "held": eval_set(sup_cv[sup_cv["subject"] == held_out])}
    n_steps = max(1, len(cpc_cv[cpc_cv["subject"].isin(train_subjects)]) // CPC_BATCH)
    return inner_val, train_subjects, sets, (min(n_steps, STEPS_CAP) if STEPS_CAP else n_steps)


def nb10_round(folder, held_out):
    """Resultado guardado de una ronda del notebook 10 (predicciones, tiempos, escalador)."""
    return json.loads((SAVED_DIR / folder / f"ronda_s{held_out:02d}_semilla0.json").read_text())


def summarize(f1):
    """Resumen entre sujetos de un vector de F1 por ronda."""
    f1 = np.asarray(f1)
    return {"f1": f1.mean(), "desv": f1.std(ddof=1) if len(f1) > 1 else 0.0, "peor": f1.min()}


# Control: las ventanas evaluadas aquí son las mismas que evaluó el notebook 10 en cada ronda
for s in FOLDS:
    assert nb10_round("GRU/h1s_lpft", s)["index"] == sup_cv.index[sup_cv["subject"] == s].tolist(), f"Ventanas distintas, sujeto {s}"
print("Las ventanas de los sujetos evaluados coinciden con las del notebook 10.")

# 4. Modelo: CPC-GRU con cabeza lineal o MLP

El modelo del notebook 11 sin cambios, salvo que `CPCClassifier` recibe el tipo de cabeza. La cabeza lineal conserva exactamente las capas del notebook 11, para poder cargar sus pesos guardados.

In [ ]:
@dataclass
class CPCConfig:
    n_channels: int = 30
    enc_channels: tuple = (32, 64, 128)
    enc_kernel_size: int = 3
    enc_dropout: float = 0.2
    ar_hidden_size: int = 256
    ar_num_layers: int = 2
    ar_dropout: float = 0.2
    n_anchors: int = N_ANCHORS
    horizons: tuple = HORIZONS

    @property
    def z_dim(self):
        return self.enc_channels[-1]

    @property
    def c_dim(self):
        return self.ar_hidden_size


class CausalEncoder(nn.Module):
    """g_enc causal: 3 bloques Conv1d -> ReLU -> Dropout, relleno sólo a la izquierda. x: (B, canales, T) -> z: (B, 128, T)"""

    def __init__(self, cfg):
        super().__init__()
        k = cfg.enc_kernel_size
        self.left_pad = k - 1
        in_channels = (cfg.n_channels,) + cfg.enc_channels[:-1]
        self.convs = nn.ModuleList([nn.Conv1d(c_in, c_out, kernel_size=k) for c_in, c_out in zip(in_channels, cfg.enc_channels)])
        self.dropout = nn.Dropout(cfg.enc_dropout)

    def forward(self, x):
        z = x
        for conv in self.convs:
            z = self.dropout(F.relu(conv(F.pad(z, (self.left_pad, 0)))))
        return z


class CPCModel(nn.Module):
    """CPC con g_ar GRU: encoder + GRU + una W_k por horizonte; pérdida InfoNCE con negativos del batch."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.encoder = CausalEncoder(cfg)
        self.ar = nn.GRU(cfg.z_dim, cfg.c_dim, num_layers=cfg.ar_num_layers, dropout=cfg.ar_dropout, batch_first=True)
        self.heads = nn.ModuleList([nn.Linear(cfg.c_dim, cfg.z_dim, bias=False) for _ in cfg.horizons])

    def encode(self, x):
        """x: (B, canales, T) -> z: (B, T, 128), c: (B, T, 256)"""
        z = self.encoder(x).transpose(1, 2)
        return z, self.ar(z)[0]

    def info_nce(self, x):
        z, c = self.encode(x)
        lo, hi = max(1, z.shape[1] // 8), z.shape[1] - max(self.cfg.horizons)   # deja sitio para leer z_{t+12}
        target = torch.arange(len(x), device=x.device)
        losses = []
        for t in torch.randint(lo, hi, size=(self.cfg.n_anchors,)).tolist():
            for head, k in zip(self.heads, self.cfg.horizons):
                logits = head(c[:, t]) @ z[:, t + k].t()                         # (B, B): la diagonal es el positivo
                losses.append(F.cross_entropy(logits, target))
        return torch.stack(losses).mean()


def make_head(kind, dim, n_classes=len(LABELS)):
    """Cabeza de clasificación sobre c_T. 'lineal': la de los notebooks 10 y 11. 'mlp': una capa oculta no lineal."""
    if kind == "lineal":
        return nn.Sequential(nn.LayerNorm(dim), nn.Dropout(0.2), nn.Linear(dim, n_classes))
    if kind == "mlp":
        return nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, MLP_HIDDEN), nn.GELU(), nn.Dropout(MLP_DROPOUT),
                             nn.Linear(MLP_HIDDEN, n_classes))
    raise ValueError(kind)


class CPCClassifier(nn.Module):
    """Encoder y GRU de CPC + cabeza (lineal o MLP) sobre c_T, el contexto del último instante."""

    def __init__(self, cpc_model, head="lineal", n_classes=len(LABELS)):
        super().__init__()
        self.encoder, self.ar = cpc_model.encoder, cpc_model.ar
        self.head = make_head(head, cpc_model.cfg.c_dim, n_classes)

    def features(self, x):
        return self.ar(self.encoder(x).transpose(1, 2))[0][:, -1]

    def forward(self, x):
        return self.head(self.features(x))


def build_cpc(n_channels, seed=SEED):
    torch.manual_seed(seed)
    return CPCModel(CPCConfig(n_channels=n_channels)).to(DEVICE)

In [ ]:
# Tamaño de cada cabeza y del resto de la red (encoder + GRU con las 10 variables de EML)
n_params = lambda m: sum(p.numel() for p in m.parameters())
_cpc = build_cpc(len(VARS))
backbone = n_params(_cpc.encoder) + n_params(_cpc.ar)
display(pd.DataFrame({"capas": [" → ".join(type(l).__name__ for l in make_head(h, 256)) for h in HEADS],
                      "parámetros de la cabeza": [n_params(make_head(h, 256)) for h in HEADS],
                      "encoder + GRU": backbone}, index=list(HEADS)).style.format(thousands=","))
del _cpc

## 4.1 LP-FT

El mismo del notebook 11. El único cambio es que `finetune` recibe el tipo de cabeza.

In [ ]:
# ----------------------------- Serie escalada de la ronda, sólo con las variables del subconjunto -----------------------------
S_T, WIN_RANGE = None, torch.arange(WIN, device=DEVICE)


def get_windows(starts):
    """Inicios de ventana -> x: (ventanas, variables del subconjunto, 48)."""
    idx = torch.as_tensor(np.asarray(starts), device=DEVICE)[:, None] + WIN_RANGE
    return S_T[:, idx].permute(1, 0, 2)


def prepare_round(held_out, groups):
    """Escalador de la ronda (comprobado contra el del notebook 10) y serie del subconjunto en el dispositivo."""
    global S_T
    inner_val, train_subjects, sets, n_steps = make_sets(held_out)
    scaler = fit_scaler(train_subjects)
    saved = nb10_round("GRU/h1s_lpft", held_out)["scaler"]
    assert all(np.allclose(saved[k], scaler[k]) for k in scaler), f"Escalador distinto al del notebook 10, sujeto {held_out}"
    cols = [CHANNELS.index(v) for v in variables_of(groups)]
    S_T = torch.from_numpy(np.ascontiguousarray(apply_scaler(scaler)[:, cols].T)).to(DEVICE)
    return {"held_out": held_out, "inner_val": inner_val, "train_subjects": train_subjects, "sets": sets,
            "n_steps": n_steps, "n_channels": len(cols)}


@torch.no_grad()
def batched(fn, starts, batch=512):
    return torch.cat([fn(get_windows(starts[s:s + batch])) for s in range(0, len(starts), batch)])


@torch.no_grad()
def predict(model, starts):
    model.eval()
    return batched(model, starts).argmax(1).cpu().numpy()


def class_weights(y):
    counts = np.bincount(y, minlength=len(LABELS))
    return torch.tensor(counts.sum() / (len(LABELS) * np.maximum(counts, 1)), dtype=torch.float32, device=DEVICE)


# ----------------------------- Preentrenamiento CPC (sin etiquetas) -----------------------------
class SameSubjectSampler:
    """Batches de CPC: ventanas al azar de UN solo sujeto, así los negativos de InfoNCE son de la misma persona."""

    def __init__(self, subjects, seed):
        self.rng = np.random.default_rng(seed)
        rows = [r for r in trials[trials["subject"].isin(subjects)].itertuples() if r.n_steps >= WIN]
        starts = np.concatenate([np.arange(r.offset, r.offset + r.n_steps - WIN + 1) for r in rows])
        subj = np.concatenate([np.full(r.n_steps - WIN + 1, r.subject) for r in rows])
        self.by_subject = {s: starts[subj == s] for s in np.unique(subj)}
        sizes = np.array([len(v) for v in self.by_subject.values()], dtype=float)
        self.p = sizes / sizes.sum()                                         # sujetos con más datos salen más seguido

    def batch(self):
        pool = self.by_subject[self.rng.choice(list(self.by_subject), p=self.p)]
        return self.rng.choice(pool, CPC_BATCH, replace=len(pool) < CPC_BATCH)


def probe_val_f1(model, rnd):
    """F1 en la validación interna de una sonda lineal (C = 1) sobre c_T. Decide la parada temprana de CPC."""
    model.eval()
    rep = lambda part: batched(lambda x: model.encode(x)[1][:, -1], rnd["sets"][part]["starts"]).float().cpu().numpy()
    probe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))
    probe.fit(rep("probe_train"), rnd["sets"]["probe_train"]["y"])
    return f1_score(rnd["sets"]["val"]["y"], probe.predict(rep("val")), average="macro")


def train_cpc(rnd, seed=SEED):
    t0 = time.time()
    model = build_cpc(rnd["n_channels"], seed)
    opt = torch.optim.Adam(model.parameters(), lr=CPC_LR)
    sampler = SameSubjectSampler(rnd["train_subjects"], seed)
    best, bad = {"f1": -1.0, "epoch": 0, "state": copy.deepcopy(model.state_dict())}, 0
    for epoch in range(1, CPC_EPOCHS + 1):
        model.train()
        for _ in range(rnd["n_steps"]):
            loss = model.info_nce(get_windows(sampler.batch()))             # pérdida contrastiva: no usa etiquetas
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
        if epoch % CPC_EVAL_EVERY == 0 or epoch == CPC_EPOCHS:
            f1_val = probe_val_f1(model, rnd)
            if f1_val > best["f1"] + 1e-4:
                best, bad = {"f1": f1_val, "epoch": epoch, "state": copy.deepcopy(model.state_dict())}, 0
            else:
                bad += 1
        if bad >= CPC_PATIENCE:
            break
    model.load_state_dict(best["state"])
    return model, {"best_epoch": best["epoch"], "f1_val": best["f1"], "minutes": (time.time() - t0) / 60}


# ----------------------------- LP-FT (con etiquetas) -----------------------------
def linear_probe_head(model, sets, seed):
    """Etapa LP: entrena sólo la cabeza con el encoder y la GRU congelados."""
    model.eval()
    X_tr, X_val = batched(model.features, sets["ft_train"]["starts"]), batched(model.features, sets["val"]["starts"])
    y_tr = torch.as_tensor(np.array(sets["ft_train"]["y"]), device=DEVICE)
    weights = class_weights(sets["ft_train"]["y"])
    opt = torch.optim.AdamW(model.head.parameters(), lr=LP_LR, weight_decay=1e-4)
    rng = np.random.default_rng(seed)
    best, bad = {"f1": -1.0, "state": copy.deepcopy(model.head.state_dict())}, 0
    for epoch in range(1, LP_EPOCHS + 1):
        model.head.train()
        order = rng.permutation(len(y_tr))
        for s in range(0, len(order), FINETUNE_BATCH):
            idx = torch.as_tensor(order[s:s + FINETUNE_BATCH], device=DEVICE)
            loss = F.cross_entropy(model.head(X_tr[idx]), y_tr[idx], weight=weights)
            opt.zero_grad(); loss.backward(); opt.step()
        model.head.eval()
        with torch.no_grad():
            f1_val = f1_score(sets["val"]["y"], model.head(X_val).argmax(1).cpu().numpy(), average="macro")
        if f1_val > best["f1"] + 1e-4:
            best, bad = {"f1": f1_val, "state": copy.deepcopy(model.head.state_dict())}, 0
        else:
            bad += 1
        if bad >= LP_PATIENCE:
            break
    model.head.load_state_dict(best["state"])
    return best["f1"]


def finetune(cpc_model, sets, head="lineal", seed=SEED):
    """LP-FT: la cabeza de LP es el punto de partida (época 0) y después se ajusta toda la red."""
    t0 = time.time()
    torch.manual_seed(seed)
    model = CPCClassifier(cpc_model, head=head).to(DEVICE)
    best = {"f1": linear_probe_head(model, sets, seed), "epoch": 0, "state": copy.deepcopy(model.state_dict())}
    opt = torch.optim.AdamW([
        {"params": list(model.encoder.parameters()) + list(model.ar.parameters()), "lr": FINETUNE_LR_BACKBONE},
        {"params": model.head.parameters(), "lr": FINETUNE_LR_HEAD}], weight_decay=1e-4)
    train = sets["ft_train"]
    weights = class_weights(train["y"])
    y_all = torch.as_tensor(np.array(train["y"]), device=DEVICE)
    rng, bad = np.random.default_rng(seed), 0
    for epoch in range(1, FT_EPOCHS + 1):
        model.train()
        order = rng.permutation(len(train["y"]))
        for s in range(0, len(order), FINETUNE_BATCH):
            idx = order[s:s + FINETUNE_BATCH]
            loss = F.cross_entropy(model(get_windows(train["starts"][idx])), y_all[idx], weight=weights)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
        f1_val = f1_score(sets["val"]["y"], predict(model, sets["val"]["starts"]), average="macro")
        if f1_val > best["f1"] + 1e-4:
            best, bad = {"f1": f1_val, "epoch": epoch, "state": copy.deepcopy(model.state_dict())}, 0
        else:
            bad += 1
        if bad >= FT_PATIENCE:
            break
    model.load_state_dict(best["state"])
    return model, {"best_epoch": best["epoch"], "f1_val": best["f1"], "minutes": (time.time() - t0) / 60}

# 5. Rondas

En cada ronda:

1. **Cabeza lineal:** se cargan los pesos finales del notebook 11. Control: sus predicciones del sujeto evaluado deben coincidir con las guardadas (≥ 99.9 % de las ventanas; la GPU puede cambiar algún empate numérico). Se calcula su F1 en entrenamiento.
2. **Cabeza MLP:** se cargan los pesos CPC del notebook 11 (antes del ajuste fino) y se corre LP-FT con la cabeza MLP. Se calcula su F1 en prueba y en entrenamiento.

El **F1 en entrenamiento** se mide con las ventanas del ajuste fino de los 10 sujetos de entrenamiento, con la red en modo evaluación (sin dropout). Cada ronda se guarda en `cabeza_<tipo>/`.

In [ ]:
def nb11_round(held_out):
    """Resultado (json) y pesos (.pt) de la ronda EML del notebook 11."""
    path = NB11_DIR / "gru_ft" / subset_key(SUBSET) / f"ronda_s{held_out:02d}"
    state = torch.load(path.with_suffix(".pt"), map_location=DEVICE)
    assert state["variables"] == VARS, f"Variables distintas en la ronda {held_out} del notebook 11"
    return json.loads(path.with_suffix(".json").read_text()), state


def cpc_from(state, n_channels):
    """CPC con los pesos preentrenados (antes del ajuste fino) de una ronda del notebook 11."""
    model = build_cpc(n_channels)
    model.load_state_dict(state["cpc"])
    return model


def run_round(held_out):
    """Cabeza lineal (recargada del notebook 11) y cabeza MLP (LP-FT nuevo) de una ronda -> {cabeza: resultado}."""
    paths = {h: RUN_DIR / f"cabeza_{h}" / f"ronda_s{held_out:02d}.json" for h in HEADS}
    if all(p.exists() for p in paths.values()):
        return {h: json.loads(p.read_text()) for h, p in paths.items()}
    rnd = prepare_round(held_out, SUBSET)
    ref, state = nb11_round(held_out)
    held, train = rnd["sets"]["held"], rnd["sets"]["ft_train"]
    assert held["index"].tolist() == ref["index"], f"Ventanas distintas a las del notebook 11, sujeto {held_out}"
    out = {}

    # Cabeza lineal: pesos finales del notebook 11
    model = CPCClassifier(cpc_from(state, rnd["n_channels"]), head="lineal").to(DEVICE)
    model.load_state_dict(state["ajuste_fino"])
    pred = predict(model, held["starts"])
    agreement = float((pred == np.array(ref["pred"])).mean())
    assert agreement >= 0.999, f"La cabeza lineal recargada no reproduce el notebook 11 (sujeto {held_out}: {agreement:.4f})"
    out["lineal"] = {"held_out": held_out, "inner_val": rnd["inner_val"], "cabeza": "lineal",
                     "f1": ref["f1"], "f1_train": f1_score(train["y"], predict(model, train["starts"]), average="macro"),
                     "acuerdo_nb11": agreement, "cpc": ref["cpc"], "ajuste_fino": ref["ajuste_fino"],
                     "index": ref["index"], "y": ref["y"], "pred": ref["pred"]}
    del model

    # Cabeza MLP: mismos pesos CPC, LP-FT nuevo
    model, info_ft = finetune(cpc_from(state, rnd["n_channels"]), rnd["sets"], head="mlp")
    pred = predict(model, held["starts"])
    out["mlp"] = {"held_out": held_out, "inner_val": rnd["inner_val"], "cabeza": "mlp",
                  "f1": f1_score(held["y"], pred, average="macro"),
                  "f1_train": f1_score(train["y"], predict(model, train["starts"]), average="macro"),
                  "cpc": ref["cpc"], "ajuste_fino": info_ft,
                  "index": held["index"].tolist(), "y": held["y"].tolist(), "pred": pred.tolist()}
    paths["mlp"].parent.mkdir(parents=True, exist_ok=True)
    torch.save({"ajuste_fino": model.state_dict(), "variables": VARS}, paths["mlp"].with_suffix(".pt"))
    del model

    for h, p in paths.items():
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(json.dumps(out[h]))
    log(f"Sujeto {held_out:2d} | lineal F1 {out['lineal']['f1']:.3f} (entr. {out['lineal']['f1_train']:.3f}) | "
        f"MLP F1 {out['mlp']['f1']:.3f} (entr. {out['mlp']['f1_train']:.3f}) | {info_ft['minutes']:.1f} min")
    return out


rounds = {s: run_round(s) for s in FOLDS}
results = {h: {k: np.array([rounds[s][h][k] for s in FOLDS]) for k in ("f1", "f1_train")} for h in HEADS}
print(f"Acuerdo mínimo de la cabeza lineal recargada con el notebook 11: {min(rounds[s]['lineal']['acuerdo_nb11'] for s in FOLDS):.4f}")

# 6. Comparativo entre las dos cabezas

## 6.1 F1 en prueba (LOSO)

F1 macro medio, desviación y peor sujeto de cada cabeza, y la comparación por sujeto de la MLP contra la lineal: diferencia de medias, en cuántos sujetos mejora y la prueba de Wilcoxon pareada.

In [ ]:
def paired(a, b):
    """Comparación por sujeto de `a` contra `b`: diferencia de medias, sujetos en que mejora y p de Wilcoxon."""
    d = np.asarray(a) - np.asarray(b)
    p = wilcoxon(a, b).pvalue if np.any(d != 0) and len(d) > 1 else np.nan
    return {"Δ vs. lineal": d.mean(), "mejora en": f"{(d > 0).sum()}/{len(d)}", "p (Wilcoxon)": p}


HEAD_NAMES = {"lineal": "Lineal (256 → 7)", "mlp": f"MLP (256 → {MLP_HIDDEN} → 7)"}
rows = []
for h in HEADS:
    s = summarize(results[h]["f1"])
    row = {"cabeza": HEAD_NAMES[h], "parámetros": n_params(make_head(h, 256)), "F1": s["f1"], "desv.": s["desv"], "peor": s["peor"]}
    if h != "lineal":
        row.update(paired(results[h]["f1"], results["lineal"]["f1"]))
    rows.append(row)
summary = pd.DataFrame(rows).set_index("cabeza")
display(summary.style.format({"parámetros": "{:,}", "F1": "{:.3f}", "desv.": "{:.3f}", "peor": "{:.3f}",
                              "Δ vs. lineal": "{:+.3f}", "p (Wilcoxon)": "{:.3f}"}, na_rep="—"))

# F1 de cada sujeto y época elegida del ajuste fino (0 = se quedó con la cabeza de la etapa LP)
per_subject = pd.DataFrame({
    "lineal": results["lineal"]["f1"], "MLP": results["mlp"]["f1"],
    "Δ (MLP − lineal)": results["mlp"]["f1"] - results["lineal"]["f1"],
    "época lineal": [rounds[s]["lineal"]["ajuste_fino"]["best_epoch"] for s in FOLDS],
    "época MLP": [rounds[s]["mlp"]["ajuste_fino"]["best_epoch"] for s in FOLDS]},
    index=pd.Index(FOLDS, name="sujeto"))
display(per_subject.style.format({"lineal": "{:.3f}", "MLP": "{:.3f}", "Δ (MLP − lineal)": "{:+.3f}"}))

## 6.2 F1 en entrenamiento contra F1 en prueba

Si a la cabeza le faltara capacidad, el F1 en **entrenamiento** de la cabeza lineal sería bajo y la MLP lo subiría, y con él el de prueba. Si el F1 en entrenamiento ya es alto, la distancia con la prueba es de generalización a sujetos nuevos.

In [ ]:
rows = []
for h in HEADS:
    tr, te = results[h]["f1_train"], results[h]["f1"]
    rows.append({"cabeza": HEAD_NAMES[h], "F1 entrenamiento": tr.mean(), "F1 prueba": te.mean(), "diferencia": (tr - te).mean(),
                 "F1 entrenamiento (peor ronda)": tr.min()})
display(pd.DataFrame(rows).set_index("cabeza").style.format("{:.3f}"))
d_train = paired(results["mlp"]["f1_train"], results["lineal"]["f1_train"])
print(f"F1 en entrenamiento, MLP contra lineal: Δ {d_train['Δ vs. lineal']:+.3f} | mejora en {d_train['mejora en']} rondas | "
      f"p {d_train['p (Wilcoxon)']:.3f}")

In [ ]:
# Izquierda: F1 en prueba de cada sujeto con las dos cabezas (líneas grises), con la media en color.
# Derecha: F1 medio en entrenamiento y en prueba de cada cabeza.
COLORS = {"lineal": "#2a6fdb", "mlp": "#d4572a"}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
ax = axes[0]
x = np.arange(len(HEADS))
per = np.array([results[h]["f1"] for h in HEADS])                       # (cabezas, sujetos)
for j in range(per.shape[1]):
    ax.plot(x, per[:, j], color="#b0b0b0", lw=1, marker="o", ms=4, zorder=1)
for xi, h in zip(x, HEADS):
    m = per[xi].mean()
    ax.scatter(xi, m, s=90, color=COLORS[h], zorder=2)
    ax.annotate(f"{m:.3f}", (xi, m), xytext=(10, 0), textcoords="offset points", va="center", fontsize=9, color="#333")
ax.set_xticks(x, [HEAD_NAMES[h] for h in HEADS])
ax.set_xlim(-0.4, len(HEADS) - 0.6)
ax.set_ylabel("F1 macro por sujeto (prueba)")
ax.set_title("F1 en prueba por sujeto")
ax.grid(axis="y", alpha=0.3)

ax = axes[1]
w = 0.38
for i, (part, key, alpha) in enumerate([("entrenamiento", "f1_train", 0.45), ("prueba", "f1", 1.0)]):
    vals = [results[h][key].mean() for h in HEADS]
    bars = ax.bar(x + (i - 0.5) * w, vals, w - 0.04, color=[COLORS[h] for h in HEADS], alpha=alpha, label=part)
    for b, v in zip(bars, vals):
        ax.annotate(f"{v:.3f}", (b.get_x() + b.get_width() / 2, v), xytext=(0, 3), textcoords="offset points",
                    ha="center", fontsize=8, color="#333")
ax.set_xticks(x, [HEAD_NAMES[h] for h in HEADS])
ax.set_ylim(0.6, 1.02)
ax.set_ylabel("F1 macro medio")
ax.set_title("Entrenamiento (claro) contra prueba (oscuro)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# 7. Conclusiones

**La cabeza MLP no mejora el desempeño de forma significativa. La cabeza lineal se conserva.**

## Resultado

| | Cabeza lineal | Cabeza MLP | Diferencia |
|---|---|---|---|
| Parámetros de la cabeza | 2,311 | 34,311 (×15) | |
| Parámetros de toda la red | 725,415 | 757,415 | +4.4 % |
| **F1 en prueba (LOSO)** | **0.873** ± 0.036 | **0.876** ± 0.025 | +0.003 · mejora en 6/12 sujetos · p = 0.62 |
| Peor sujeto | 0.775 (sujeto 2) | 0.810 (sujeto 2) | |
| **F1 en entrenamiento** | **0.897** | **0.900** | +0.003 · mejora en 7/12 rondas · p = 0.52 |
| Distancia entrenamiento − prueba | 0.024 | 0.023 | |

Control: la cabeza lineal recargada reprodujo el 100 % de las predicciones del notebook 11.

## Por qué no vale la pena cambiar la cabeza lineal

**1. La mejora en prueba no se distingue del azar.**
- La diferencia media es +0.003.
- La MLP gana en 6 sujetos y pierde en 6.
- La diferencia de un sujeto a otro varía ± 0.019, unas seis veces más que la mejora media.
- Hay sujetos que suben (el 2, de 0.775 a 0.810; el 1, de 0.857 a 0.886) y otros que bajan (el 7, de 0.889 a 0.859; el 4, de 0.908 a 0.897). Es la variación esperada al repetir un ajuste fino, no una mejora sistemática.

**2. Quince veces más capacidad no sube el F1 en entrenamiento. Esta es la prueba directa.**
Si a la cabeza le faltara capacidad para separar las clases, la MLP clasificaría mejor al menos las ventanas con las que se entrena. No lo hace: 0.897 frente a 0.900, sin diferencia significativa.

El límite no está en la frontera de decisión. Está en la información que llega a la cabeza: el vector `c_T`, y antes que eso las 10 variables de la ventana.

**3. Las dos cabezas fallan en las mismas ventanas.**
- Las dos coinciden en el 95.7 % de sus predicciones.
- El 9.3 % de las ventanas lo fallan ambas.
- Solo el 2.2 % lo falla únicamente la lineal, y el 1.9 % únicamente la MLP.

Los errores son propiedad de los datos, no de la cabeza.

**4. Las clases difíciles siguen igual.**

| Clase | Lineal | MLP |
|---|---|---|
| PT | 0.821 | 0.828 |
| LT | 0.823 | 0.823 |
| LF | 0.837 | 0.845 |
| N, K, W y PF | — | cambian menos de 0.004 |

La confusión de PT y LT con N (colocar o levantar con poca variación en las flexiones) no se resuelve con una frontera no lineal.

**5. No hay sobreajuste que aprovechar ni que corregir.**
La distancia entre entrenamiento y prueba es pequeña (≈ 0.02) con las dos cabezas. El modelo se comporta casi igual con sujetos nuevos que con los de entrenamiento. Los dos se quedan en ≈ 0.90.

Ese techo compartido apunta a ventanas ambiguas por naturaleza: ventanas de 4 s que cruzan una transición entre actividades y se etiquetan con el instante final. También apunta a actividades que, con estas variables, se ven casi iguales. Una cabeza más grande no puede separar lo que la entrada no distingue.

Matiz: la parada temprana elige la época con mejor F1 en validación, no la de mejor ajuste en entrenamiento. El F1 en entrenamiento es el del modelo que se usa, no el máximo que la red podría memorizar. La regla es la misma para las dos cabezas, así que la comparación es justa.

**6. La MLP agrega complejidad sin beneficio.**
Suma 32,000 parámetros y dos hiperparámetros más (ancho de la capa oculta y dropout), que habría que ajustar y justificar. Por parsimonia, a igual desempeño se prefiere el modelo más simple.

La cabeza lineal tiene además una ventaja de interpretación: si una sola capa lineal basta, el trabajo de separar las actividades ya lo hicieron el preentrenamiento CPC y el ajuste fino. Es la forma estándar de evaluar representaciones aprendidas.

## Qué sí podría mover el desempeño

El resultado redirige el esfuerzo de la cabeza hacia la **entrada** del modelo:

- **Información de la ventana completa:** la cabeza solo ve el último instante (`c_T`). Un *pooling* sobre los 48 pasos (promedio o atención) le da otra información, no solo más capacidad.
- **Información de carga:** la sEMG distingue colocar o levantar un objeto de estar quieto, que es justamente la confusión PT/LT con N.
- **Más sujetos:** el F1 depende del sujeto evaluado (0.78–0.91). Más diversidad de personas en el entrenamiento ataca el problema de generalización directamente.